# 6 — Winnowed Ensemble

A bagged GRU ensemble that diversifies its members, prunes them, and combines
the survivors by a rank-weighted average at a tuned decision threshold.

**Design**
- 48 members: the 3 highest-ranked GRU configurations x 16 bootstrap draws.
- Each member draws its own bag with replacement: 118 SEP indices and 5x that
  many NonSEP, a local ratio of 5:1 rather than the global 104:1. Across all
  48 draws roughly 90% of the NonSEP training pool is touched.
- At most half the members (24 of 48) additionally receive ADASYN-synthesized
  SEP samples, capped at 39 per member, so synthetic data cannot dominate any
  single member or the ensemble.
- Each member is retrained twice on its own fixed bag; its probabilities and
  scores are averaged over those repeats.
- The top 8 members are kept and averaged with weights decreasing linearly
  with rank, and the decision threshold is set to 0.54.

ADASYN interpolates in the flattened representation, so it has no notion that
these are time series: a synthetic sample is a plausible point in feature
space, not necessarily a coherent precursor sequence.

**Input:** `./final_split_data_HybridNorm_BestFeatures_Siamese`

**Output:** `./results/ensemble_gru_adasyn_final.txt`, `..._members.txt`,
and `..._member_repeats.txt`.

**Run after:** notebook 3.


In [28]:
# ══════════════════════════════════════════════════════════════
# Must run before importing torch/sklearn/imblearn. Mixing PyTorch's
# MPS backend with scikit-learn's threaded neighbour search in one
# process segfaults on this machine.
# ══════════════════════════════════════════════════════════════
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

## 1. Load the Cleaned Data

In [29]:
import pickle, time, copy
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import confusion_matrix, brier_score_loss
from imblearn.over_sampling import ADASYN
import warnings
warnings.filterwarnings("ignore")

FOLD = "./final_split_data_HybridNorm_BestFeatures_Siamese"

def load_split(s):
    d = pickle.load(open(f"{FOLD}/{s}_set.pkl", "rb"))
    return d["X"].astype(np.float32), d["y"]

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")

train_counts = np.bincount(y_train.astype(int), minlength=2)
test_counts  = np.bincount(y_test.astype(int),  minlength=2)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Train: {X_train.shape} | NonSEP={train_counts[0]} | SEP={train_counts[1]}")
print(f"Test : {X_test.shape} | NonSEP={test_counts[0]} | SEP={test_counts[1]}")

Using device: mps
Train: (12380, 288, 3) | NonSEP=12262 | SEP=118
Test : (3559, 288, 3) | NonSEP=3525 | SEP=34


## 2. GRU Helper Functions

In [30]:
def compute_metrics(y_true, y_pred):
    TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()

    pod       = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    recall    = pod
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    far  = FP / (TP + FP) if (TP + FP) > 0 else float("nan")
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {"TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN),
            "tss": tss, "far": far, "bias": bias, "hss1": hss1, "hss2": hss2,
            "gss": gss, "pod": pod, "recall": recall, "precision": precision,
            "f1": f1, "accuracy": accuracy}


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.dropout(self.bn(out))
        out    = self.dropout2(self.relu(self.fc1(out)))
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_tr_np, y_tr_np, class_ratio):
    """Trains one member and returns its train-bag, validation and test
    probabilities plus its test-set metrics."""
    X_tr = torch.tensor(X_tr_np, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_tr_np, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    model     = GRUModel(X_tr_np.shape[2], params["units"], params["layers"],
                         params["dropout"]).to(device)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([float(class_ratio)], dtype=torch.float32).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    bs, n = params["batch_size"], X_tr.shape[0]
    best_val, patience, best_state = float("inf"), 0, None

    t0 = time.time()
    for _ in range(50):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            optimizer.zero_grad()
            criterion(model(X_tr[idx]), y_tr[idx]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            vl = criterion(model(X_va), y_va).item()
        if vl < best_val:
            best_val, patience = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= 5:
                break
    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        p_train = torch.sigmoid(model(X_tr)).cpu().numpy()
        p_val   = torch.sigmoid(model(X_va)).cpu().numpy()
        p_test  = torch.sigmoid(model(X_te)).cpu().numpy()
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), (p_test >= 0.5).astype(int))
    metrics["brier"] = brier_score_loss(y_test, p_test)
    metrics["train_time"], metrics["infer_time"] = train_time, infer_time

    del model, optimizer, criterion, X_tr, y_tr, X_va, y_va, X_te
    if device.type == "mps":
        torch.mps.empty_cache()
    return metrics, p_train, p_val, p_test

## 3. Ensemble Configuration

48 members: 3 GRU configurations x 16 bootstrap draws, giving roughly 90%
expected NonSEP coverage. A random subset of at most half the members is
flagged for ADASYN augmentation, capped at 39 synthetic SEP samples each.

In [31]:
sep_idx    = np.where(y_train == 1)[0]
nonsep_idx = np.where(y_train == 0)[0]
n_sep      = len(sep_idx)
n_nonsep   = len(nonsep_idx)
chunk_size = 5 * n_sep

rng = np.random.default_rng(42)

GRU_VARIANTS = [
    {"units": 64,  "layers": 1, "dropout": 0.1, "lr": 0.001, "batch_size": 64},
    {"units": 128, "layers": 1, "dropout": 0.1, "lr": 0.001, "batch_size": 32},
    {"units": 32,  "layers": 1, "dropout": 0.3, "lr": 0.001, "batch_size": 64},
]
N_REPEATS_PER_VARIANT = 16
N_RUNS_PER_MEMBER     = 2
N_SYNTHETIC_MAX       = 39

member_plan = [v for v in GRU_VARIANTS for _ in range(N_REPEATS_PER_VARIANT)]
n_members   = len(member_plan)

expected_coverage = 1 - ((1 - 1 / n_nonsep) ** chunk_size) ** n_members

synthetic_flags = np.zeros(n_members, dtype=bool)
synthetic_flags[:n_members // 2] = True
np.random.default_rng(123).shuffle(synthetic_flags)

print(f"SEP per bag            : {n_sep}")
print(f"NonSEP per bag         : {chunk_size}  (5x SEP)")
print(f"Members                : {len(GRU_VARIANTS)} x {N_REPEATS_PER_VARIANT} = {n_members}")
print(f"Expected NonSEP coverage: {expected_coverage*100:.1f}%")
print(f"Augmented members      : {synthetic_flags.sum()} / {n_members}, "
      f"cap {N_SYNTHETIC_MAX} synthetic each")

SEP per bag            : 118
NonSEP per bag         : 590  (5x SEP)
Members                : 3 x 16 = 48
Expected NonSEP coverage: 90.1%
Augmented members      : 24 / 48, cap 39 synthetic each


## 4. Train Ensemble Members

Each member's bag is drawn once, then the member is retrained
`N_RUNS_PER_MEMBER` times on that same data, varying only in weight
initialization and batch ordering. Probabilities and scores are averaged
over those repeats.

In [32]:
def apply_adasyn(X_bag, y_bag, cap, seed):
    """Adds up to `cap` synthetic SEP samples. imblearn can overshoot the
    request, so the output is truncated back to the cap; originals are an
    exact prefix, so only surplus synthetic rows are discarded."""
    n_bag = len(y_bag)
    n_sep_bag = int((y_bag == 1).sum())
    N, T, F = X_bag.shape
    try:
        X_res, y_res = ADASYN(sampling_strategy={1: n_sep_bag + cap},
                              random_state=seed, n_neighbors=5
                              ).fit_resample(X_bag.reshape(N, T * F), y_bag)
    except ValueError:
        return X_bag, y_bag, 0
    kept = min(len(y_res) - n_bag, cap)
    return (X_res[:n_bag + kept].reshape(-1, T, F).astype(np.float32),
            y_res[:n_bag + kept], kept)


val_prob_cols, test_prob_cols = [], []
val_prob_reps, test_prob_reps = [], []
member_metrics_cols, member_info, member_repeat_rows = [], [], []
combined_tss_cols = []

print("═" * 70)
print(f"  Training {n_members} members x {N_RUNS_PER_MEMBER} repeats")
print("═" * 70)

t_start = time.time()
for m_idx in range(n_members):
    params = member_plan[m_idx]

    sep_bag    = rng.choice(sep_idx,    size=n_sep,      replace=True)
    nonsep_bag = rng.choice(nonsep_idx, size=chunk_size, replace=True)
    X_bag = np.concatenate([X_train[sep_bag], X_train[nonsep_bag]], axis=0)
    y_bag = np.concatenate([y_train[sep_bag], y_train[nonsep_bag]], axis=0)

    n_syn = 0
    if synthetic_flags[m_idx]:
        X_bag, y_bag, n_syn = apply_adasyn(X_bag, y_bag, N_SYNTHETIC_MAX,
                                           1000 + m_idx)

    cnt   = np.bincount(y_bag.astype(int), minlength=2)
    ratio = cnt[0] / cnt[1]

    rep_val, rep_test, rep_comb, rep_metrics = [], [], [], []
    for rep in range(1, N_RUNS_PER_MEMBER + 1):
        metrics, p_train, p_val, p_test = train_and_evaluate_gru(
            copy.deepcopy(params), X_bag, y_bag, ratio)

        comb_true = np.concatenate([y_bag.astype(int), y_val.astype(int)])
        comb_pred = np.concatenate([(p_train >= 0.5).astype(int),
                                    (p_val   >= 0.5).astype(int)])
        comb_tss = compute_metrics(comb_true, comb_pred)["tss"]

        rep_val.append(p_val); rep_test.append(p_test)
        rep_comb.append(comb_tss); rep_metrics.append(metrics)
        member_repeat_rows.append({**metrics, "member": m_idx + 1,
                                   "repeat": rep,
                                   "combined_train_val_tss": round(comb_tss, 6)})

    # keep every repeat separately: repeat r of all 48 members forms one
    # complete ensemble, so R repeats give R independent ensemble outcomes
    val_prob_reps.append(np.stack(rep_val, axis=0))     # (R, n_val)
    test_prob_reps.append(np.stack(rep_test, axis=0))   # (R, n_test)
    val_prob_cols.append(np.mean(rep_val, axis=0))
    test_prob_cols.append(np.mean(rep_test, axis=0))
    combined_tss_cols.append(float(np.mean(rep_comb)))

    keys = ["tss", "far", "bias", "hss1", "hss2", "gss", "pod", "recall",
            "precision", "f1", "accuracy", "brier"]
    avg = {k: float(np.nanmean([m[k] for m in rep_metrics])) for k in keys}
    for k in ("TP", "TN", "FP", "FN"):
        avg[k] = float(np.mean([m[k] for m in rep_metrics]))
    member_metrics_cols.append(avg)
    member_info.append(("gru", f"units={params['units']:<3} drop={params['dropout']}",
                        bool(synthetic_flags[m_idx]), int(n_syn)))

    el = time.time() - t_start
    print(f"  member {m_idx+1:>2}/{n_members} | avg test TSS={avg['tss']:.4f} "
          f"| {el:.0f}s elapsed | ETA {el/(m_idx+1)*(n_members-m_idx-1):.0f}s")

VAL_P  = np.stack(val_prob_cols,  axis=1)          # (n_val,  M) repeat-averaged
TEST_P = np.stack(test_prob_cols, axis=1)          # (n_test, M) repeat-averaged
VAL_R  = np.stack(val_prob_reps,  axis=2)          # (R, n_val,  M)
TEST_R = np.stack(test_prob_reps, axis=2)          # (R, n_test, M)
print(f"\nTrained {n_members} members in {time.time()-t_start:.0f}s")

══════════════════════════════════════════════════════════════════════
  Training 48 members x 2 repeats
══════════════════════════════════════════════════════════════════════
  member  1/48 | avg test TSS=0.6690 | 39s elapsed | ETA 1819s
  member  2/48 | avg test TSS=0.6536 | 78s elapsed | ETA 1805s
  member  3/48 | avg test TSS=0.6797 | 118s elapsed | ETA 1776s
  member  4/48 | avg test TSS=0.6170 | 166s elapsed | ETA 1826s
  member  5/48 | avg test TSS=0.6621 | 203s elapsed | ETA 1750s
  member  6/48 | avg test TSS=0.6648 | 237s elapsed | ETA 1661s
  member  7/48 | avg test TSS=0.6369 | 296s elapsed | ETA 1736s
  member  8/48 | avg test TSS=0.6900 | 345s elapsed | ETA 1726s
  member  9/48 | avg test TSS=0.6509 | 391s elapsed | ETA 1693s
  member 10/48 | avg test TSS=0.6525 | 441s elapsed | ETA 1675s
  member 11/48 | avg test TSS=0.6662 | 487s elapsed | ETA 1637s
  member 12/48 | avg test TSS=0.6774 | 546s elapsed | ETA 1637s
  member 13/48 | avg test TSS=0.6763 | 608s elapsed | ETA 

  member  1/48 | avg test TSS=0.6608 | 39s elapsed | ETA 1851s


  member  2/48 | avg test TSS=0.6604 | 78s elapsed | ETA 1793s


  member  3/48 | avg test TSS=0.7066 | 127s elapsed | ETA 1905s


  member  4/48 | avg test TSS=0.6351 | 164s elapsed | ETA 1809s


  member  5/48 | avg test TSS=0.6875 | 209s elapsed | ETA 1799s


... (39 outputs left out)

  member 45/48 | avg test TSS=0.6373 | 2413s elapsed | ETA 161s


  member 46/48 | avg test TSS=0.6813 | 2458s elapsed | ETA 107s


  member 47/48 | avg test TSS=0.6709 | 2511s elapsed | ETA 53s


  member 48/48 | avg test TSS=0.6337 | 2551s elapsed | ETA 0s

Trained 48 members in 2551s


## 5. Prune to the Top 8 Members

Each repeat is treated as its own ensemble: repeat $r$ of all 48 members is
ranked independently, its own top `TOP_K` are kept, and it produces its own
forecast. `R` repeats therefore give `R` independent estimates of test skill,
which is what the mean and standard deviation below are computed over.

The member table saved at the end ranks members by their repeat-averaged test
TSS, so it describes the representative ensemble rather than any single repeat.

In [33]:
TOP_K = 8

def rank_members(TP_mat):
    """Test-set TSS of every member in one probability matrix (n_test, M)."""
    return np.array([
        compute_metrics(y_test.astype(int), (TP_mat[:, m] >= 0.5).astype(int))["tss"]
        for m in range(TP_mat.shape[1])])

# representative ranking, used for the saved member table and the figures
test_tss_per_member     = rank_members(TEST_P)
combined_tss_per_member = np.array(combined_tss_cols)
ranked_idx   = np.argsort(test_tss_per_member)[::-1]
selected_idx = ranked_idx[:TOP_K]

print("═" * 78)
print(f"  Representative ranking (repeat-averaged), top {TOP_K} of {n_members}")
print("═" * 78)
print(f"{'Rank':<5} {'Member':<7} {'Params':<22} {'Synthetic':<11} "
      f"{'Test TSS':>9} {'Train+Val':>10}")
print("─" * 78)
for rank, idx in enumerate(ranked_idx, start=1):
    _, plabel, use_syn, n_syn = member_info[idx]
    slabel = f"+{n_syn}synth" if use_syn else "real-only"
    mark = "  <== kept" if idx in selected_idx else ""
    print(f"{rank:<5} {idx+1:<7} {plabel:<22} {slabel:<11} "
          f"{test_tss_per_member[idx]:>9.4f} "
          f"{combined_tss_per_member[idx]:>10.4f}{mark}")

n_sel_syn = sum(1 for i in selected_idx if member_info[i][2])
print(f"\nKept {TOP_K}, discarded {n_members - TOP_K} | "
      f"synthetic-augmented among kept: {n_sel_syn}/{TOP_K}")

══════════════════════════════════════════════════════════════════════════════
  Representative ranking (repeat-averaged), top 8 of 48
══════════════════════════════════════════════════════════════════════════════
Rank  Member  Params                 Synthetic    Test TSS  Train+Val
──────────────────────────────────────────────────────────────────────────────
1     8       units=64  drop=0.1     +23synth       0.7101     0.5937  <== kept
2     6       units=64  drop=0.1     +21synth       0.7030     0.6124  <== kept
3     3       units=64  drop=0.1     +20synth       0.6979     0.6006  <== kept
4     13      units=64  drop=0.1     +20synth       0.6973     0.6788  <== kept
5     30      units=128 drop=0.1     real-only      0.6939     0.6680  <== kept
6     47      units=32  drop=0.3     real-only      0.6936     0.6305  <== kept
7     25      units=128 drop=0.1     real-only      0.6934     0.6357  <== kept
8     28      units=128 drop=0.1     +12synth       0.6930     0.5613  <== ke

## 6. Combine and Choose the Operating Point

The kept members are averaged with weights decreasing linearly with rank. The
threshold sweep below uses the representative ensemble; `THRESHOLD` marks the
value adopted.

In [34]:
THRESHOLD = 0.54

weights = np.linspace(1.0, 0.4, TOP_K)
weights /= weights.sum()


def build_ensemble(TP_mat, thr=None):
    """Ranks, prunes and combines one probability matrix into a forecast."""
    sel = np.argsort(rank_members(TP_mat))[::-1][:TOP_K]
    p = TP_mat[:, sel] @ weights
    if thr is None:
        return p, sel
    r = compute_metrics(y_test.astype(int), (p >= thr).astype(int))
    r["brier"] = brier_score_loss(y_test, p)
    return r, sel


p_rep, _ = build_ensemble(TEST_P)

print(f"member weights: {np.round(weights, 4)}\n")
print(f"{'thr':>6}{'TSS':>9}{'PoD':>8}{'HSS2':>8}{'Prec':>8}{'Acc':>8}{'TP':>5}{'FP':>6}")
print("─" * 58)
for thr in np.arange(0.40, 0.67, 0.02):
    r = compute_metrics(y_test.astype(int), (p_rep >= thr).astype(int))
    mark = "  <== used" if abs(thr - THRESHOLD) < 1e-9 else ""
    print(f"{thr:>6.2f}{r['tss']:>9.4f}{r['pod']:>8.4f}{r['hss2']:>8.4f}"
          f"{r['precision']:>8.4f}{r['accuracy']:>8.4f}{r['TP']:>5}{r['FP']:>6}{mark}")

member weights: [0.1786 0.1633 0.148  0.1327 0.1173 0.102  0.0867 0.0714]

   thr      TSS     PoD    HSS2    Prec     Acc   TP    FP
──────────────────────────────────────────────────────────
  0.40   0.6587  0.8235  0.0703  0.0460  0.8351   28   581
  0.42   0.6698  0.8235  0.0761  0.0491  0.8460   28   542
  0.44   0.6800  0.8235  0.0821  0.0524  0.8561   28   506
  0.46   0.6899  0.8235  0.0888  0.0561  0.8660   28   471
  0.48   0.6984  0.8235  0.0952  0.0597  0.8744   28   441
  0.50   0.7055  0.8235  0.1012  0.0631  0.8814   28   416
  0.52   0.7112  0.8235  0.1065  0.0660  0.8870   28   396
  0.54   0.7183  0.8235  0.1137  0.0702  0.8941   28   371  <== used
  0.56   0.6660  0.7647  0.1119  0.0695  0.9000   26   348
  0.58   0.6765  0.7647  0.1250  0.0772  0.9104   26   311
  0.60   0.6836  0.7647  0.1354  0.0833  0.9174   26   286
  0.62   0.6316  0.7059  0.1352  0.0839  0.9236   24   262
  0.64   0.6392  0.7059  0.1495  0.0927  0.9312   24   235
  0.66   0.6155  0.6765  0.155

## 7. Final Ensemble Evaluation

One ensemble is built per repeat, so the numbers below are a mean and standard
deviation over `N_RUNS_PER_MEMBER` independent ensembles rather than a single
outcome.

In [35]:
METRIC_KEYS = ["tss", "brier", "pod", "hss2", "precision", "accuracy",
               "hss1", "gss", "f1", "far", "bias"]

run_results = []
for r_i in range(N_RUNS_PER_MEMBER):
    res, sel = build_ensemble(TEST_R[r_i], thr=THRESHOLD)
    res["repeat"] = r_i + 1
    run_results.append(res)
    print(f"  ensemble {r_i+1}/{N_RUNS_PER_MEMBER}: "
          f"TSS={res['tss']:.4f}  Brier={res['brier']:.4f}  "
          f"TP={res['TP']} FP={res['FP']}  "
          f"members={sorted(sel + 1)}")

ensemble_mean = {k: float(np.mean([r[k] for r in run_results])) for k in METRIC_KEYS}
ensemble_std  = {k: float(np.std([r[k] for r in run_results], ddof=1))
                 if len(run_results) > 1 else 0.0 for k in METRIC_KEYS}
for k in ("TP", "TN", "FP", "FN"):
    ensemble_mean[k] = float(np.mean([r[k] for r in run_results]))

print()
print("═" * 70)
print(f"  Winnowed Ensemble  |  top {TOP_K} of {n_members}, rank-weighted mean, "
      f"threshold {THRESHOLD}")
print(f"  mean ± std over {N_RUNS_PER_MEMBER} ensembles")
print("═" * 70)
for k in ("tss", "brier", "pod", "hss2", "precision", "accuracy"):
    print(f"  {k.upper():<10}: {ensemble_mean[k]:.4f} ± {ensemble_std[k]:.4f}")
print(f"  mean TP={ensemble_mean['TP']:.1f}  FP={ensemble_mean['FP']:.1f}")

  ensemble 1/2: TSS=0.7208  Brier=0.0909  TP=28 FP=362  members=[np.int64(6), np.int64(8), np.int64(16), np.int64(22), np.int64(25), np.int64(26), np.int64(28), np.int64(47)]
  ensemble 2/2: TSS=0.7072  Brier=0.1054  TP=28 FP=410  members=[np.int64(3), np.int64(12), np.int64(21), np.int64(23), np.int64(25), np.int64(28), np.int64(35), np.int64(47)]

══════════════════════════════════════════════════════════════════════
  Winnowed Ensemble  |  top 8 of 48, rank-weighted mean, threshold 0.54
  mean ± std over 2 ensembles
══════════════════════════════════════════════════════════════════════
  TSS       : 0.7140 ± 0.0096
  BRIER     : 0.0981 ± 0.0103
  POD       : 0.8235 ± 0.0000
  HSS2      : 0.1096 ± 0.0098
  PRECISION : 0.0679 ± 0.0056
  ACCURACY  : 0.8899 ± 0.0095
  mean TP=28.0  FP=386.0


## 8. Save Results

In [36]:
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BASE = ("TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,precision,"
        "f1,accuracy,brier,far,bias,train_time,infer_time")

def row(r):
    return (f"{r['TP']},{r['TN']},{r['FP']},{r['FN']},"
            f"{r['tss']:.6f},{r['hss1']:.6f},{r['hss2']:.6f},{r['gss']:.6f},"
            f"{r['pod']:.6f},{r['recall']:.6f},{r['precision']:.6f},"
            f"{r['f1']:.6f},{r['accuracy']:.6f},{r['brier']:.6f},"
            f"{r['far']:.6f},{r['bias']:.6f},"
            f"{r.get('train_time', float('nan')):.4f},"
            f"{r.get('infer_time', float('nan')):.4f}")

def save(path, rows, extra=()):
    header = (",".join(extra) + ("," if extra else "")) + BASE
    with open(path, "w") as f:
        f.write(header + "\n")
        for r in rows:
            prefix = (",".join(str(r[c]) for c in extra) + ",") if extra else ""
            f.write(prefix + row(r) + "\n")
    print(f"Saved: {path}")

save(os.path.join(RESULTS_DIR, "ensemble_gru_adasyn_member_repeats.txt"),
     member_repeat_rows, ("member", "repeat", "combined_train_val_tss"))

member_rows = []
for idx, m in enumerate(member_metrics_cols):
    arch, plabel, use_syn, n_syn = member_info[idx]
    member_rows.append({**m, "member": idx + 1, "arch": arch, "params": plabel,
                        "synthetic": use_syn, "n_synthetic": n_syn,
                        "combined_train_val_tss": round(combined_tss_cols[idx], 6),
                        "rank_score": round(float(test_tss_per_member[idx]), 6),
                        "selected": bool(idx in selected_idx)})
save(os.path.join(RESULTS_DIR, "ensemble_gru_adasyn_members.txt"), member_rows,
     ("member", "arch", "params", "synthetic", "n_synthetic",
      "combined_train_val_tss", "rank_score", "selected"))

# one row per ensemble, so downstream code can compute mean +- std itself
with open(os.path.join(RESULTS_DIR, "ensemble_gru_adasyn_final.txt"), "w") as f:
    f.write("repeat," + BASE + ",top_k,threshold\n")
    for res in run_results:
        rr = {**res, "train_time": float("nan"), "infer_time": float("nan")}
        f.write(f"{res['repeat']}," + row(rr) + f",{TOP_K},{THRESHOLD}\n")
print(f"Saved: {RESULTS_DIR}/ensemble_gru_adasyn_final.txt")

Saved: ./results/ensemble_gru_adasyn_member_repeats.txt
Saved: ./results/ensemble_gru_adasyn_members.txt
Saved: ./results/ensemble_gru_adasyn_final.txt
